In [ ]:
### 연락처로 인기유저

In [2]:
!pip install koreanize_matplotlib

import warnings
import koreanize_matplotlib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import ast

# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

try:
	from google.colab import drive
	drive.mount('/content/drive')

	import os
	os.chdir('/content/drive/MyDrive/파트4')
	print('✅ Succesful access google_drive_directory')

except Exception as e:
	print('❌')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 20.0 MB/s eta 0:00:00
Mounted at /content/drive
✅ Succesful access google_drive_directory


In [6]:
API_KEY_PATH = '/content/drive/MyDrive/파트4/sprintda03-yujin.json'

## get_df 함수
def get_df(db_name, table_name):
    table_name = pd.read_csv(
        f"gs://high_project/{db_name}/{table_name}.csv",
        storage_options={'token' : API_KEY_PATH}
        )
    return table_name


## literal_eval 형변환 함수
import ast
def to_literal_eval(df, column):
    df[column] = df[column].apply(lambda x: ast.literal_eval(x) if x != '[]' else [])
    return df

drop_users = [831956, 1580627, 1580689, 1580626, 995177]

In [4]:
accounts_user_contacts = get_df('votes','accounts_user_contacts')
accounts_user_contacts = accounts_user_contacts[['id', 'user_id', 'invite_user_id_list', 'contacts_count']]
to_literal_eval(accounts_user_contacts, 'invite_user_id_list')

In [37]:
for i in drop_users:
    count_drop_rows = len(accounts_user_contacts[accounts_user_contacts['invite_user_id_list'].apply(lambda x: 831956 in x)])
    if count_drop_rows != 0:
        print(f"‼️ 관리자 {i}가 포함된 행 {count_drop_rows}개 존재")
    else:
        print(f"✅ 관리자 {i} 포함행 없음")

✅ 관리자 831956 포함행 없음
✅ 관리자 1580627 포함행 없음
✅ 관리자 1580689 포함행 없음
✅ 관리자 1580626 포함행 없음
✅ 관리자 995177 포함행 없음


In [39]:
accounts_user_contacts = accounts_user_contacts[~accounts_user_contacts['user_id'].isin(drop_users)]

,id,user_id,invite_user_id_list,contacts_count
0,259,1167696,[],30
1,1756,863169,[],79
2,13742,857205,[854615],21
3,13754,851431,[],29
4,13756,855476,[849318],28
...,...,...,...,...
5058,12981327,1480714,[],7
5059,13391623,1506575,[],1
5060,14465598,1577436,[],0
5061,14579987,1582145,[],0


In [30]:
accounts_user_contacts

,id,user_id,invite_user_id_list,contacts_count
0,259,1167696,[],30
1,1756,863169,[],79
2,13742,857205,[854615],21
3,13754,851431,[],29
4,13756,855476,[849318],28
...,...,...,...,...
5058,12981327,1480714,[],7
5059,13391623,1506575,[],1
5060,14465598,1577436,[],0
5061,14579987,1582145,[],0


In [ ]:
accounts_user_contacts['id'].nunique() # 5063
accounts_user_contacts['user_id'].nunique() # 5063

5063

In [ ]:
accounts_user_contacts['invite_user_id_list'] = accounts_user_contacts['invite_user_id_list'].apply(ast.literal_eval) # literal_eval 으로변환
expended_contacts = accounts_user_contacts.explode('invite_user_id_list').reset_index(drop=True) # 데이터 프래암 형식 변환

In [ ]:
expended_contacts
# 유저id
# 유저 전화번호를 가지고 있는 유저수
# 해당 유저id를 초대했던 유저 id

,id,user_id,invite_user_id_list,contacts_count
0,259,1167696,NaN,30
1,1756,863169,NaN,79
2,13742,857205,854615,21
3,13754,851431,NaN,29
4,13756,855476,849318,28
...,...,...,...,...
5587,12981327,1480714,NaN,7
5588,13391623,1506575,NaN,1
5589,14465598,1577436,NaN,0
5590,14579987,1582145,NaN,0


In [ ]:
accounts_user_contacts['user_id'].duplicated().sum()

0

In [ ]:
expended_contacts[['user_id', 'invite_user_id_list']].duplicated().sum()

0

## 유저별 초대수

In [ ]:
accounts_user_contacts.query('user_id==1207606')

,id,user_id,invite_user_id_list,contacts_count
1009,593562,1207606,"[1050526, 1050496, 1167888, 1185543, 1186531, 853318, 976805, 1119808, 1166297, 1063007]",62


In [ ]:
expended_contacts['user_id'].value_counts().reset_index()

,user_id,count
0,1207606,10
1,932308,9
2,1122686,7
3,1042593,6
4,1138054,6
...,...,...
5058,1100842,1
5059,1145467,1
5060,887222,1
5061,872289,1


In [ ]:
fig = px.box(expended_contacts['user_id'].value_counts().reset_index(), x="count", title="친구 초대수")
fig.show()

## 유저별 초대 받은수

In [ ]:
expended_contacts['invite_user_id_list'].value_counts().reset_index()

,invite_user_id_list,count
0,1154585,21
1,1233225,17
2,883602,16
3,1041381,14
4,873001,14
...,...,...
1117,875729,1
1118,872532,1
1119,870562,1
1120,890315,1


In [ ]:
fig = px.box(expended_contacts['invite_user_id_list'].value_counts().reset_index(), x="count", title="초대 받은수")
fig.show()

fig = px.histogram(expended_contacts['invite_user_id_list'].value_counts().reset_index(), x="count",  title="초대한 횟수(count) 분포")
fig.show()

In [ ]:
fig = px.box(expended_contacts['user_id'].value_counts().reset_index(), x="count", title="친구 초대수")
fig.show()

In [ ]:
fig = px.box(accounts_user_contacts, x="contacts_count", title="앱사용자 연락처 보유수 분포 vs 폰내 연락처 보유수")
fig.show()